## Build tools (search + calculator + local summarizer)
- langchain 0.3.7
- langchain-core 0.3.15
- langchain-community 0.3.2
- langgraph 0.2.20

### - langchain             (core LLM framework)
### - langgraph             (workflow / agent orchestration)
### - ddg-search tool       (web search through DuckDuckGo)
### - HuggingFaceHub LLM    (no API key fee required)

In [ ]:
#Sets your HuggingFace token for accessing free hosted inference models.
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "your_token"

In [2]:

from langchain_community.llms import HuggingFaceEndpoint

## Create an LLM endpoint object that will call Mistral 7B Instruct.
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    temperature=0,
    max_new_tokens=512,      # optional 
    repetition_penalty=1.1,  # optional
)


C:\Users\mohit\AppData\Local\Temp\ipykernel_24064\4225686523.py:4: LangChainDeprecationWarning: The class `HuggingFaceEndpoint` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEndpoint``.
  llm = HuggingFaceEndpoint(
c:\Users\mohit\Documents\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import re, textwrap, json ## Standard libraries used for parsing, formatting, and cleaning text.

# LangChain community integrations:
# HuggingFaceHub → connects to free public models on Hugging Face (no endpoint permissions needed)
# DuckDuckGoSearchResults → web search tool returning real snippets
# tool → decorator to define callable AI tools 
from langgraph.graph import StateGraph, START, END # For full agent workflows
from langgraph.checkpoint.memory import MemorySaver
from langchain.tools import tool
#from langchain_community.tools.ddg_search import DuckDuckGoSearchResults
from langchain_community.tools.ddg_search.tool import DuckDuckGoSearchRun ##  DuckDuckGo web search wrapper
from langchain_community.llms import HuggingFaceHub





In [ ]:
from ddgs import DDGS # DuckDuckGo Search library for direct search without API
import time # for simulating delays in real-world agent interactions
# dictionary for simple in-memory context between user turns
# MEMORY: holds previous numeric results or context so queries can build on each other
#pip install ddgs --upgrade

memory_store = {}

# A helper function to perform a live search using the DDGS library

def live_ddgs_search(query, max_results=5): 
    time.sleep(2)  # reduce rate-limit risk

    with DDGS(timeout=20) as ddgs: #initialize the DDGS search context
        results = ddgs.text(
            query,
            max_results=max_results,
            region="de-de",
            safesearch="moderate",
            backend="auto" # use "html" for real snippets, "api" for metadata (default auto-detects based on query)
        )
    return list(results)

# ───────────────────────────────────────────
# INITIALIZE THE LANGUAGE MODEL
#
# HuggingFaceHub allows you to call models such as Mistral, Falcon, etc.
# Here we use Mistral 7B Instruct – a strong open model with zero temperature
# (temperature=0 means deterministic, no randomness in output)
llm = HuggingFaceHub(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    model_kwargs={"temperature": 0, "max_new_tokens": 512}
)

# ───────────────────────────────────────────
# DEFINE TOOLS
# ───────────────────────────────────────────
# 1. Search Tool – fetches top 5 web results as JSON-like objects
#search = DuckDuckGoSearchResults(max_results=5)


# 2. Calculator Tool – evaluates math expressions safely
@tool("calculator")
def calculator(expr: str) -> str:
    """Safely evaluate arithmetic expressions like 4.4 * 0.05."""
    # Ensure only digits and math symbols are allowed
    if not re.fullmatch(r"[0-9\.\+\-\*\/\(\) ]+", expr):
        return "Invalid expression"
    try:
        # Use Python's eval to compute numeric result
        return str(eval(expr))
    except Exception as e:
        # Return error string if evaluation fails
        return f"Error: {e}"

# 3. Summarizer Tool – takes a paragraph and shortens it
@tool("summarizer")
def summarizer(text: str) -> str:
    """Summarize text into 1–2 concise sentences."""
    # Split into sentences and remove blanks
    sents = [s.strip() for s in text.split(".") if s.strip()]
    # Keep only the first two sentences for a short summary
    short = ". ".join(sents[:2])
    # Wrap lines neatly to 80 chars per line
    return textwrap.fill(short, width=80)

# ───────────────────────────────────────────
# MAIN AGENT FUNCTION
# ───────────────────────────────────────────
def agent_query(query: str, thread_id: str = "default"):
    """
    Pipeline:
    1) Receive query
    2) Add previous memory if exists
    3) Search the web
    4) Extract numeric values from text
    5) Compute 5%
    6) Summarize result
    7) Save memory
    """

    # ——— Print formatted header
    print("═" * 90)
    print(f" User Query: {query}")
    print("─" * 90)

    # ——— Load any previous context for continuity
    if thread_id in memory_store:
        print(f"Context: {memory_store[thread_id]}")
        # Append context to query to remind the LLM of earlier data
        query = f"{query} (consider previous context: {memory_store[thread_id]})"

    # STEP 1️. — SEARCH
    #result_raw = search.run(query)
    results = live_ddgs_search(query, max_results=3)
    print(results)


    # DuckDuckGoSearchResults can return either a JSON string or a list of dicts.
    if isinstance(results, str):
        try:
            results = json.loads(results)
        except json.JSONDecodeError:
            print("Error parsing search results. Raw output:")
            print(results)
            results = []

    # STEP 2. — DISPLAY TOP SNIPPETS NEATLY
    print("Search Results:")
    clean_text = ""  # We'll accumulate all snippet text for analysis
    for i, r in enumerate(results):  # only first 1 for clarity
        body = r.get("body", "")
        title = r.get("title", "")
        link = r.get("href", "")

        clean_text += title + " " + body + " "
        print(f"  • {title}")  # show title
        print(f"    {textwrap.fill(body, width=80)}")  # show wrapped snippet
        if link:
            print(f"    {link}\n")  # show URL
        clean_text += title + " " + body + " "
        

    # STEP 3️. — EXTRACT NUMERIC VALUE(S)
    # Find all patterns like "4.4" or "123"
    nums = re.findall(r"\d+(?:\.\d+)?", clean_text)
    # Filter out obvious non-GDP numbers like years
    nums = [float(n) for n in nums]

# remove likely years
    nums = [n for n in nums if n < 1900 or n > 2100]

    if not nums:
        print(" No numeric GDP value detected in the snippets.")
        print("═" * 90)
        return

    # Take the largest number as most likely the GDP figure
    value = max(nums)

    # STEP 4️. — CALCULATE 5% OF GDP
    calc = calculator.run(f"{value} * 0.05")

    # STEP 5️. — SUMMARIZE EVERYTHING
    summary_text = f"{clean_text.strip()}. 5% of that is approximately {calc}."
    summary = summarizer.run(summary_text)

    # STEP 6️. — DISPLAY RESULTS
    print("Calculation:")
    print(f"   Base value: {value}")
    print(f"   5% Value  : {calc}\n")

    print("Summary:")
    print(textwrap.fill(summary, width=80))
    print("═" * 90)

    # STEP 7️. — UPDATE MEMORY
    memory_store[thread_id] = f"{value} trillion USD GDP remembered."

# ───────────────────────────────────────────
# RUN EXAMPLES
# ───────────────────────────────────────────
# Example 1: Independent search for Germany
agent_query("Germany nominal GDP 2025 trillion USD IMF", thread_id="demo")

# Example 2: Follows the previous memory context (thread_id="demo")
# Demonstrates persistence of conversation
agent_query("France nominal GDP 2025 trillion USD IMF.", thread_id="demo")


══════════════════════════════════════════════════════════════════════════════════════════
 User Query: Germany nominal GDP 2025 trillion USD IMF
──────────────────────────────────────────────────────────────────────────────────────────
[{'title': 'Wirtschaft Deutschlands - Wikipedia', 'href': 'https://de.wikipedia.org/wiki/Wirtschaft_Deutschlands', 'body': '54.989 USD (nominal) (2024) 70.878 USD (PPP) (2024). BIP nach ... 2025). Arbeitslosenquote, 6,4 % (Aug. 2025). Außenhandel. Export, 1,9 Billionen ...'}, {'title': 'Germany: 2025 Article IV Consultation-Press Release; Staff Report', 'href': 'https://www.imf.org/-/media/files/publications/cr/2026/english/1deuea2026001.pdf', 'body': '6 Feb 2026 ... The debt-to-GDP ratio is estimated by IMF staff to have ... Nominal GDP (billions of euros). 4219.3. 4329.0. 4470.5. 4624.6. 4797.3.'}, {'title': "Germany is richer than ever. But most people aren't. Over the past ...", 'href': 'https://www.instagram.com/p/DXtQ5gTlNvo/', 'body': "29 Apr 202

NameError: name 'snippet' is not defined